In [14]:
import pandas as pd
import os

os.getcwd()

'/Users/degn400/Git_Repos/DancePartner/vignettes'

In [ ]:
import re

def __get_ome_df(ome_path: str, delim: str = ",", ome_type: str = None):
    '''
    Pull an omes files, and parse the file to be a pandas dataframe 

    Parameters
    ----------
    omes_path
        The path to the omes file formatted with the first column as the ID and the second as the Synonyms
    
    delim
        The delimiter for the file. Default is a ',' for a csv. 

    ome_type
        An optional identifier for the ome type
    '''

    # Read the ome csv
    ome = pd.read_csv(ome_path, sep = delim)

    # Instantiate the ome dictionary
    ome_dict = {}

    for row in range(len(ome)):
        
        terms = str(ome["Synonyms"][row]).split("; ")
        terms = [re.sub(r'[^a-zA-Z0-9]', '', term.strip().lower()) for term in terms]

        if not isinstance(terms, list):
            terms = list(terms)

        for do_not_use in ["", "nan"]:
            if do_not_use in terms:
                terms.remove(do_not_use)

        # Only return if there is one match
        if len(terms) > 0:
            ome_dict[ome.iloc[row, 0]] = list(set(terms))

    # Make dictionary
    df = pd.DataFrame(ome_dict.items()).explode(1).rename({0: "ID", 1: "Synonym"}, axis = 1)
    if ome_type is not None:
        df["Ome"] = ome_type

    return df

In [29]:
lipidome = __get_ome_df("../omes/LipidMaps_Lipidome.csv", ome_type = "lipid")
metabolome = __get_ome_df("../omes/CHEBI_metabolome.txt", "\t", "metabolite")
proteome = __get_ome_df("../../edge_weighting_truthdata/omes/UP000000803_proteome.txt", "\t", "gene product")
genome = __get_ome_df("../../edge_weighting_truthdata/omes/7227_genome.txt", "\t", "gene product")

In [47]:
ome_data = pd.concat([lipidome, metabolome, proteome, genome])

# Remove terms that do no fit the requirements
stopwords = pd.read_csv(os.path.join("../omes/stop_words_english.txt"))["stopwords"].tolist()

# Determine all terms that meet requirements
unique_terms = list(set(ome_data["Synonym"].to_list()))
cleaned_terms = [term for term in unique_terms if len(term) >= 3 and len(term) <= 50 and term not in stopwords]

# Filter ome data down to cleaned terms 
ome_data = ome_data[ome_data["Synonym"].isin(cleaned_terms)]

In [48]:
ome_data = ome_data.groupby("Synonym").agg({"ID": list, "Ome": list}).reset_index()
ome_data

,Synonym,ID,Ome
0,00xf3,[CHEBI:53456],[metabolite]
1,01289,"[A0A0B4K7H1, A0A0B4K819, A1Z6P2, E1JGY5, E1JGY...","[gene product, gene product, gene product, gen..."
2,0129,[CHEBI:73908],[metabolite]
3,03659,[A0A0B4KEG2],[gene product]
4,04053,[Q9VNR6],[gene product]
...,...,...,...
214350,zzzz7111417icosatetraenoicacid,[LMFA01031069],[lipid]
214351,zzzzeicosa8111417tetraenoate,[CHEBI:71563],[metabolite]
214352,zzzzeicosa8111417tetraenoicacid,[CHEBI:71488],[metabolite]
214353,zzzzicosa8111417tetraenoate,[CHEBI:71563],[metabolite]


In [44]:
def __get_label_priority(ome_list: list):
    '''
    From a list of any omes, rank the order by lipid, metabolite, and gene product
    '''
    
    if "lipid" in ome_list:
        return "lipid"
    elif "metabolite" in ome_list:
        return "metabolite"
    else:
        return "gene product"

In [53]:
ome_data["Ome"] = [__get_label_priority(ome_list) for ome_list in ome_data["Ome"]]

synonym_table = ome_data
synonym_table

,Synonym,ID,Ome
0,00xf3,[CHEBI:53456],metabolite
1,01289,"[A0A0B4K7H1, A0A0B4K819, A1Z6P2, E1JGY5, E1JGY...",gene product
2,0129,[CHEBI:73908],metabolite
3,03659,[A0A0B4KEG2],gene product
4,04053,[Q9VNR6],gene product
...,...,...,...
214350,zzzz7111417icosatetraenoicacid,[LMFA01031069],lipid
214351,zzzzeicosa8111417tetraenoate,[CHEBI:71563],metabolite
214352,zzzzeicosa8111417tetraenoicacid,[CHEBI:71488],metabolite
214353,zzzzicosa8111417tetraenoate,[CHEBI:71563],metabolite


In [ ]:
# Load output from UniProt

